In [0]:
-------------------------------------------------------------------------
-- DELTA LAKE OPTIMIZATION & MAINTENANCE
-- Purpose: Implements VACUUM, OPTIMIZE (Z-Order), and demonstrates Time Travel
-- Rubric: Domain 4 (Delta Features)
-- -------------------------------------------------------------------------

-- 1. OPTIMIZE GOLD FACT TABLE (Performance)
-- We Z-ORDER by 'event_date' and 'brand' because those are the most common filters in our Dashboard.
-- This colocates related data in the same files, making queries 10x faster.
SELECT "Optimizing Gold Fact Table...";

OPTIMIZE ecommerce_analytics_dev.gold_layer.gold_fact_sales
ZORDER BY (event_date, brand);

-- 2. OPTIMIZE GOLD DIMENSION (SCD2 Performance)
-- We Z-ORDER by 'product_id' to make joining faster.
SELECT " Optimizing Gold Dimension Table...";

OPTIMIZE ecommerce_analytics_dev.gold_layer.gold_dim_products
ZORDER BY (product_id);

-- 3. VACUUM SILVER & GOLD (Cost Savings)
-- Removes files older than 7 days (standard retention) that are no longer needed.
-- Note: This prevents Time Travel beyond 7 days to save storage.
SELECT " Vacuuming Silver & Gold Tables...";

VACUUM ecommerce_analytics_dev.silver_layer.events_silver RETAIN 168 HOURS; -- 7 Days
VACUUM ecommerce_analytics_dev.gold_layer.gold_fact_sales RETAIN 168 HOURS;
VACUUM ecommerce_analytics_dev.gold_layer.gold_dim_products RETAIN 168 HOURS;

SELECT " Delta Lake Maintenance Complete.";

###Evidence of Time Travel 

In [0]:
DESCRIBE HISTORY ecommerce_analytics_dev.gold_layer.gold_fact_sales;

###Time Travel

In [0]:
-- "Show me the count of rows as of Version 0 (The first run)"
SELECT COUNT(*) as count_v0 
FROM ecommerce_analytics_dev.gold_layer.gold_fact_sales VERSION AS OF 0;

-- Check Version 2 (The first actual data commit)
SELECT COUNT(*) as count_v1
FROM ecommerce_analytics_dev.gold_layer.gold_fact_sales VERSION AS OF 2;

In [0]:
DESCRIBE HISTORY ecommerce_analytics_dev.gold_layer.gold_fact_sales;

In [0]:
-- EVIDENCE 1: PROOF OF OPTIMIZE & VACUUM (The Audit Trail)
-- Look for "OPTIMIZE" and "VACUUM" in the 'operation' column.
-- This proves the maintenance job actually ran.
SELECT 
    version, 
    timestamp, 
    operation, 
    operationMetrics.numFilesAdded, 
    operationMetrics.numFilesRemoved
FROM (DESCRIBE HISTORY ecommerce_analytics_dev.gold_layer.gold_fact_sales)
WHERE operation IN ('OPTIMIZE', 'VACUUM', 'STREAMING UPDATE')
ORDER BY version DESC
LIMIT 10;

In [0]:
-- EVIDENCE 2: PROOF OF TIME TRAVEL (Functional Test)
-- This query proves you can access data from a specific previous version.
-- We compare the current version vs. Version 1 (or 0).
SELECT 
    'Current Version' as source,
    COUNT(*) as row_count 
FROM ecommerce_analytics_dev.gold_layer.gold_fact_sales
UNION ALL
SELECT 
    'Version 1 (Historical)' as source,
    COUNT(*) as row_count 
FROM ecommerce_analytics_dev.gold_layer.gold_fact_sales VERSION AS OF 1;

In [0]:
-- EVIDENCE 3: PROOF OF PHYSICAL STORAGE (Detail)
-- This shows the file format is 'delta' and lists the number of files.
DESCRIBE DETAIL ecommerce_analytics_dev.gold_layer.gold_fact_sales;

|format|id|name|description|location|createdAt|lastModified|partitionColumns|clusteringColumns|numFiles|sizeInBytes|properties|minReaderVersion|minWriterVersion|tableFeatures|statistics|clusterByAuto|
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
|delta|86a97dd5-d71b-42cc-a932-db89a55458bc|ecommerce_analytics_dev.gold_layer.gold_fact_sales|Transactional Fact Table. One row per event. Contains Raw Price.|abfss://unity-catalog-storage@saecommerceojas.dfs.core.windows.net/__unitystorage/catalogs/c6352481-cba8-40a2-94ad-ee5f69250e1f/tables/f25ba946-348d-4af0-b39c-769233c017c5|2026-02-03T13:36:53.441+00:00|2026-02-04T12:04:26.000+00:00|[]|[]|6|1910336209|{"quality":"gold","spark.internal.pipelines.reconciliation_query_without_timetravel":"SELECT * FROM `ecommerce_analytics_dev`.`gold_layer`.`__materialization_mat_94607052_7b75_4d6a_86e7_b235e96246c9_gold_fact_sales_1`","pipeline_internal.catalogType":"UNITY_CATALOG","delta.enableChangeDataFeed":"true","spark.sql.internal.pipelines.parentTableId":"4866f9b7-ae04-46e2-af7d-737b9587b1ec","delta.enableDeletionVectors":"true","pipelines.pipelineId":"94607052-7b75-4d6a-86e7-b235e96246c9","pipeline_internal.enzymeMode":"Advanced","spark.internal.streaming_table.parentTable":"`ecommerce_analytics_dev`.`gold_layer`.`gold_fact_sales`","delta.enableRowTracking":"true","delta.rowTracking.materializedRowCommitVersionColumnName":"_row-commit-version-col-60af5545-d983-4a17-a357-7911fae6b199","pipelines.metastore.tableName":"`ecommerce_analytics_dev`.`gold_layer`.`__materialization_mat_94607052_7b75_4d6a_86e7_b235e96246c9_gold_fact_sales_1`","delta.rowTracking.materializedRowIdColumnName":"_row-id-col-b22b45f6-6746-4f3c-8725-ad8e336160ed"}|3|7|["appendOnly","changeDataFeed","deletionVectors","domainMetadata","invariants","rowTracking"]|{"numRowsDeletedByDeletionVectors":0,"numDeletionVectors":0}|false|